# Classification and representative evaluation

This notebook reads the archived grouped evaluation and operational model settings. It constructs the 17 predictors from the development spectra and checks the human reference totals. Model fitting itself is performed by the terminal model or complete route.

In [ ]:
from pathlib import Path
import os,sys,json
import pandas as pd
from IPython.display import display,SVG
from terraclass_ppi.paths import CODE_ROOT as ROOT,output_root
from terraclass_ppi.data_io import prepare_reference,table_directory,iter_candidate_groups
WORK=Path(os.environ['RBC_OUTPUT']).resolve() if os.environ.get('RBC_OUTPUT') else prepare_reference()
os.environ['RBC_OUTPUT']=str(WORK)
FIGURES=output_root()/'figures'
from terraclass_ppi.spatial import CFG


## Evaluate candidates and the selected representatives

The predictors are ten reflectances divided by the spectrum’s Euclidean norm, six spectral indices and log(1+norm). Five outer partitions group candidates by class and date to evaluate the procedure. Four inner partitions choose among four tree configurations and select the threshold with the highest sensitivity subject to positive precision ≥0.95 and at least 30 accepted examples.

Each outer model classifies every candidate in its reserved groups. The accepted candidate with the highest PPI is then selected; a tie is resolved by the lowest canonical pixel index. Human judgments assess those selected representatives. The final operational model is fitted on 2,422 development examples. The 58 evaluation-only judgments remain outside that fit. The 95% compatibility target applies to the overall representative evaluation.

In [ ]:
from terraclass_ppi.spectral import atributos
training=pd.read_csv(WORK/'model/training.csv')
X,names=atributos(training);assert X.shape==(2422,17)
display(pd.Series(names,name='Spectral predictors'))
display(pd.read_csv(WORK/'evaluation/candidate_external_metrics.csv'))
reference=pd.read_csv(WORK/'evaluation/reference.csv')
display(reference.groupby('classe').evaluation_label.value_counts())
assert reference.evaluation_label.eq('compatible').sum()==502
print('External representative compatibility: 502/519 = 96.72%')
display(json.loads((WORK/'model/model.json').read_text())['choice'])


## Read the two evaluations

Candidate metrics describe predictions on labeled examples reserved from each outer fit. The representative reference describes the output of classification followed by highest-PPI selection: 502 of 519 representatives were judged compatible (96.72%). The printed operational choice comes from internal selection on the full development base; it is used to build the time series.

## References

Friedman (2001), gradient boosting; Pedregosa et al. (2011), scikit-learn (full citations in `REFERENCES.md`). Cawley and Talbot (2010): https://www.jmlr.org/papers/v11/cawley10a.html . Threshold selection: https://scikit-learn.org/1.6/modules/classification_threshold.html